# NB07 — Rust Fundamentals for Computational Biology

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

Rust is a systems programming language — compiled, statically typed, with no garbage collector — that achieves C-level performance. What distinguishes it from C is its **ownership model**: the compiler enforces memory safety at compile time, eliminating a whole class of bugs (use-after-free, data races, null pointer dereferences) without runtime overhead.

Why does this matter for computational biology? The noodles library (BAM/CRAM/VCF parsing, used at Sanger), needletail (FASTA/FASTQ parsing, used at One Codex), rust-bio (alignment algorithms), and sourmash (metagenomic sketching) are all written in Rust. Reading and writing Rust is becoming a practical skill in high-throughput bioinformatics.

---

## 2. Intuition

In Python, variables are **reference-counted objects** — the runtime tracks how many references to each object exist, and frees it when the count reaches zero. This is safe and convenient, but adds overhead (every assignment touches a reference count) and can cause pause-inducing garbage collection cycles.

Rust's **ownership model** is the compiler's alternative: every value has exactly one owner. When the owner goes out of scope, the value is freed — deterministically, with no GC, no reference counting, no runtime overhead. The compiler enforces this statically, so if you write code that would use a freed value or share it unsafely across threads, it **refuses to compile**.

This notebook introduces Rust syntax through side-by-side comparison with Python. Rust code is shown as strings (the notebook is Python); Python equivalents are fully executable.

---

## 3. Biological background

A FASTA file for the human genome is ~3 GB. Reading it in Python naively loads it all into memory. In Rust with noodles or needletail, parsing is **streaming** — one record at a time, no intermediate heap allocation for the whole file.

This matters when you are:
- Counting k-mers in a 100 GB metagenomic dataset
- Computing GC-content histograms over 50M reads in a FASTQ file
- Indexing a 50 GB BAM file with random-access queries

Rust's zero-overhead abstractions let you write the readable code and get the performance of hand-optimized C.

---

## 4. Mathematical explanation — The type system

Rust's type system assigns every value a type at compile time. Types carry:
- **Size**: `u8` is 1 byte, `f64` is 8 bytes — known at compile time, no boxing
- **Semantics**: `usize` is the platform-native unsigned integer for indexing
- **Ownership rules**: whether the type is Copy (cheap to duplicate) or Move (transferred on assignment)

| Rust type | Python equivalent | Notes |
|-----------|------------------|-------|
| `i32` | `int` | 32-bit signed integer |
| `f64` | `float` | 64-bit floating point |
| `bool` | `bool` | true/false |
| `String` | `str` | heap-allocated, growable UTF-8 |
| `&str` | (view of str) | string slice, borrowed reference |
| `Vec<T>` | `list` | heap-allocated, growable sequence |
| `HashMap<K,V>` | `dict` | hash map |
| `Option<T>` | `T | None` | value or absence of value |
| `Result<T,E>` | (exception) | value or error — explicit handling |

---

## 5. Computational explanation — Key syntax

```rust
// Variables: immutable by default
let x = 5;           // x: i32, inferred
let mut y = 5;       // mutable
y += 1;              // OK

// Functions
fn add(a: i32, b: i32) -> i32 {
    a + b              // no semicolon = return value (expression)
}

// Structs (like Python dataclasses)
struct Sequence { id: String, bases: Vec<u8> }

// impl: methods on structs
impl Sequence {
    fn gc_content(&self) -> f64 {
        let gc = self.bases.iter()
            .filter(|&&b| b == b'C' || b == b'G')
            .count();
        gc as f64 / self.bases.len() as f64
    }
}

// Enums + match (exhaustive pattern matching)
enum Base { A, T, C, G }
let base = Base::A;
match base {
    Base::A | Base::T => println!("AT"),
    Base::C | Base::G => println!("GC"),
}
```

---

## 6. Python implementation — side-by-side illustrations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("NB07: Rust Fundamentals")
print("Rust code is shown as strings. Python equivalents run in this notebook.")

In [ ]:
# --- Rust: variables and functions ---
rust_variables = '''
// Rust: immutable by default
fn main() {
    let x: i32 = 42;          // immutable, type annotated
    let y = 3.14_f64;         // type inferred from literal suffix
    let mut z = 0i32;         // mutable
    z += x;                   // OK: z is mut
    // x = 100;               // ERROR: x is immutable
    
    let seqs: Vec<String> = vec![                     // Vec<String> = list of strings
        String::from("ATCG"),
        String::from("GCTA"),
    ];
    println!("{:?}", seqs);    // debug print
}
'''

print("Rust: variables and types")
print(rust_variables)

# Python equivalent (fully executable)
print("\nPython equivalent:")
x: int = 42          # type hint only — not enforced
y: float = 3.14
z = 0
z += x               # Python has no immutability for variables
seqs: list[str] = ["ATCG", "GCTA"]
print(f"  x={x}, y={y}, z={z}")
print(f"  seqs={seqs}")
print("\nKey difference: Rust immutability is ENFORCED by the compiler.")
print("Python 'final' is a convention, not a constraint.")

In [ ]:
# --- Rust: structs and impl blocks ---
rust_struct = '''
// Rust: Sequence struct with methods
#[derive(Debug)]                           // auto-generate debug printing
struct Sequence {
    id: String,
    bases: Vec<u8>,                        // bases as bytes (ASCII)
}

impl Sequence {
    // Constructor: associated function (no &self)
    fn new(id: &str, bases: &str) -> Sequence {
        Sequence {
            id: id.to_string(),
            bases: bases.bytes().collect(),
        }
    }
    
    // Method: takes &self (immutable borrow of the struct)
    fn gc_content(&self) -> f64 {
        let gc = self.bases.iter()
            .filter(|&&b| b == b\'C\' || b == b\'G\')
            .count();
        gc as f64 / self.bases.len() as f64
    }
    
    fn len(&self) -> usize {
        self.bases.len()
    }
}

fn main() {
    let seq = Sequence::new("seq1", "ATCGCGAT");
    println!("{}: GC={:.2}", seq.id, seq.gc_content());
    // Output: seq1: GC=0.38
}
'''

print("Rust: Sequence struct")
print(rust_struct)

# Python equivalent
from dataclasses import dataclass

@dataclass
class Sequence:
    id: str
    bases: bytes
    
    @classmethod
    def new(cls, id: str, bases: str) -> 'Sequence':
        return cls(id=id, bases=bases.encode('ascii'))
    
    def gc_content(self) -> float:
        gc = sum(1 for b in self.bases if chr(b) in ('C', 'G'))
        return gc / len(self.bases)
    
    def __len__(self) -> int:
        return len(self.bases)

seq = Sequence.new('seq1', 'ATCGCGAT')
print(f"\nPython: {seq.id}: GC={seq.gc_content():.2f}")

In [ ]:
# --- Rust: enums and match ---
rust_enum = '''
// Rust: enum for nucleotide bases
#[derive(Debug, PartialEq)]
enum Base {
    A, T, C, G, N,   // N = ambiguous
}

impl Base {
    fn from_char(c: char) -> Base {
        match c {
            \'A\' | \'a\' => Base::A,
            \'T\' | \'t\' => Base::T,
            \'C\' | \'c\' => Base::C,
            \'G\' | \'g\' => Base::G,
            _            => Base::N,   // _ = catch-all (like Python else:)
        }
    }
    
    fn is_gc(&self) -> bool {
        matches!(self, Base::C | Base::G)   // concise pattern match
    }
}

// Rust match is EXHAUSTIVE: you must handle every variant.
// If you add a new variant to Base, every match on Base becomes
// a compile error until you add the new arm.
'''

print("Rust: enum and match")
print(rust_enum)

# Python equivalent
from enum import Enum

class Base(Enum):
    A = 'A'; T = 'T'; C = 'C'; G = 'G'; N = 'N'
    
    @classmethod
    def from_char(cls, c: str) -> 'Base':
        # Python match is NOT exhaustive — missing cases silently return None
        match c.upper():
            case 'A': return cls.A
            case 'T': return cls.T
            case 'C': return cls.C
            case 'G': return cls.G
            case _:   return cls.N
    
    def is_gc(self) -> bool:
        return self in (Base.C, Base.G)

for char in ['A', 'C', 'X']:
    b = Base.from_char(char)
    print(f"  '{char}' → {b.name}, is_gc={b.is_gc()}")

print("\nKey difference: Rust match is exhaustive. Missing arms = compile error.")
print("Python match is not exhaustive — missing cases silently fall through.")

In [ ]:
# --- Rust: Vec<T> and HashMap<K,V> ---
rust_collections = '''
use std::collections::HashMap;

fn main() {
    // Vec<T>: growable array, like Python list
    let mut v: Vec<i32> = Vec::new();
    v.push(1); v.push(2); v.push(3);
    let third = v[2];         // index access (panics on out-of-bounds)
    let safe = v.get(10);     // returns Option<&i32>: Some(&val) or None
    
    // HashMap<K,V>: like Python dict
    let mut kmer_counts: HashMap<String, usize> = HashMap::new();
    
    let seq = "ATCGATCG";
    let k = 3;
    for i in 0..seq.len() - k + 1 {
        let kmer = &seq[i..i+k];                   // string slice, no allocation
        *kmer_counts.entry(kmer.to_string())        // get or insert entry
            .or_insert(0)                          // insert 0 if absent
            += 1;                                  // increment
    }
    
    for (kmer, count) in &kmer_counts {
        println!("{}: {}", kmer, count);
    }
}
'''

print("Rust: Vec and HashMap")
print(rust_collections)

# Python equivalent
from collections import defaultdict

seq = "ATCGATCG"
k = 3
kmer_counts: dict[str, int] = defaultdict(int)
for i in range(len(seq) - k + 1):
    kmer = seq[i:i+k]
    kmer_counts[kmer] += 1

print("\nPython k-mer counts:")
for kmer, count in sorted(kmer_counts.items()):
    print(f"  {kmer}: {count}")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Memory layout of Vec vs Python list
ax = axes[0]
ax.axis('off')
mem_text = (
    "Memory Layout\n\n"
    "Python list [1, 2, 3]:\n"
    "  list object\n"
    "  ├── ptr → [ptr1, ptr2, ptr3]  (array of pointers)\n"
    "  ├── len = 3\n"
    "  └── capacity = 4\n"
    "       │       │       │\n"
    "       v       v       v\n"
    "   int obj  int obj  int obj\n"
    "   (28 B)   (28 B)   (28 B)\n"
    "   Total: 28*3 + overhead = ~120 B\n\n"
    "Rust Vec<i32> vec![1, 2, 3]:\n"
    "  Vec struct\n"
    "  ├── ptr → [1i32, 2i32, 3i32]  (array of values)\n"
    "  ├── len = 3\n"
    "  └── capacity = 3\n"
    "       (4 B each, contiguous)\n"
    "   Total: 4*3 + 24 = ~36 B\n\n"
    "Rust stores values directly.\n"
    "Python stores pointers to objects."
)
ax.text(0.02, 0.98, mem_text, transform=ax.transAxes,
        fontsize=8.5, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('Vec<i32> vs Python list:\nMemory Layout', fontweight='bold')

# Panel 2: Ownership transfer (move semantics)
ax = axes[1]
ax.axis('off')
own_text = (
    "Rust Ownership (Move Semantics)\n\n"
    "let s1 = String::from(\"ATCG\");\n"
    "  s1 owns the heap data\n\n"
    "let s2 = s1;    // MOVE\n"
    "  s2 now owns the data\n"
    "  s1 is INVALID (use = compile error)\n\n"
    "let s3 = s2.clone();  // COPY\n"
    "  s3 owns a new copy\n"
    "  s2 still valid (owns original)\n\n"
    "Python (reference counting):\n"
    "s1 = 'ATCG'\n"
    "s2 = s1       # both refer to same obj\n"
    "               # refcount = 2\n"
    "del s1        # refcount = 1, not freed\n"
    "del s2        # refcount = 0, freed\n\n"
    "Rust: deterministic drop.\n"
    "Python: GC decides when."
)
ax.text(0.02, 0.98, own_text, transform=ax.transAxes,
        fontsize=8.5, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#fef9e7', alpha=0.8))
ax.set_title('Ownership: Move vs Clone\nvs Python Reference Counting', fontweight='bold')

# Panel 3: Compile/run workflow
ax = axes[2]
ax.axis('off')
workflow_text = (
    "Rust Compile/Run Workflow\n\n"
    "1. Write src/main.rs\n"
    "   (or src/lib.rs for a library)\n\n"
    "2. cargo build\n"
    "   Compiles to native binary\n"
    "   cargo build --release: optimized\n\n"
    "3. cargo run\n"
    "   Build + run in one step\n\n"
    "4. cargo test\n"
    "   Run all #[test] functions\n\n"
    "5. cargo clippy\n"
    "   Linter — catches common mistakes\n\n"
    "Key files:\n"
    "  Cargo.toml    dependencies, metadata\n"
    "  src/main.rs   executable entry point\n"
    "  src/lib.rs    library (for PyO3)\n\n"
    "No separate 'pip install' for deps:\n"
    "  cargo handles everything."
)
ax.text(0.02, 0.98, workflow_text, transform=ax.transAxes,
        fontsize=8.5, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eafaf1', alpha=0.8))
ax.set_title('Rust Cargo Workflow', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb07_rust_fundamentals.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Python translation:** For each Rust construct shown, write the Python equivalent and note one way they differ in behavior (not just syntax):
   - `let mut v: Vec<f64> = Vec::with_capacity(1000);`
   - `match value { Some(x) => println!("{}", x), None => println!("empty") }`
   - `for (i, base) in seq.iter().enumerate() { ... }`

2. **Result type:** In Rust, `Result<T, E>` replaces exceptions. Write the Python equivalent of:
   ```rust
   fn parse_gc(s: &str) -> Result<f64, String> {
       if s.is_empty() { return Err("empty input".to_string()); }
       let gc = s.chars().filter(|&c| c == 'C' || c == 'G').count();
       Ok(gc as f64 / s.len() as f64)
   }
   ```
   Use Python's `try/except` or a `Result`-like pattern with `tuple[float | None, str | None]`.

3. **Struct design:** Design a `FastaRecord` Python dataclass that mirrors the Rust struct from this notebook. Add: `reverse_complement()`, `is_valid()` (checks only ATCGN bases), and `__len__`. Test it on 5 sequences.

## 12. Reflection

*In your own words: What problem does Rust's ownership model solve that Python's GC also solves, but differently? What is the key trade-off between the two approaches? When would you choose Rust over Python for a bioinformatics task?*

---

## Papers referenced

- Kryuchkova-Mostacci et al. (2025). "A Survey of Bioinformatics Tools in Rust." — read after NB10.

## References

- The Rust Book: https://doc.rust-lang.org/book/ (Chapter 1–8 covers this notebook)
- Rust by Example: https://doc.rust-lang.org/rust-by-example/
- noodles (Rust bioinformatics): https://github.com/zaeleus/noodles

## Future work / open questions

- Rust enums can carry data: `enum Record { Fasta(FastaRecord), Fastq(FastqRecord) }`. How is this different from Python's `Union[FastaRecord, FastqRecord]` type hint?
- The Rust standard library's `BufReader` streams file content in chunks. How does this compare to Python's `open(file).readline()` approach in terms of memory usage?